# 🎙️ ASR Shootout — Benchmark Notebook
### Deepgram vs Whisper vs IndicConformer vs Sarvam
**For Indian Conversational Speech (Bangalore Locality Names)**

---

**Runtime:** GPU (T4 recommended) — Runtime → Change Runtime Type → T4 GPU

**Run order:** Execute cells top to bottom. Each section is independent after setup.

**Before running:**
1. Upload your audio files to Google Drive at `My Drive/asr-shootout/recordings/`
2. Have your Deepgram and Sarvam API keys ready
3. Enable GPU runtime above

## 🔧 Section 0: Setup & Installation
Run once at the start of every session.

In [ ]:
# Clone your GitHub repo (update URL after pushing)
import os

REPO_URL = 'https://github.com/YOUR_USERNAME/asr-shootout.git'  # ← UPDATE THIS
REPO_DIR = '/content/asr-shootout'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
    print('✅ Repo cloned')
else:
    !git -C {REPO_DIR} pull
    print('✅ Repo updated')

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

In [ ]:
# Install all dependencies
print('Installing dependencies... (this takes 2-3 minutes)')

!pip install -q deepgram-sdk>=6.0.0
!pip install -q faster-whisper>=1.0.0
!pip install -q jiwer>=3.0.3
!pip install -q rapidfuzz>=3.5.0
!pip install -q jellyfish>=1.0.0
!pip install -q librosa>=0.10.1 soundfile>=0.12.1 pydub>=0.25.1
!pip install -q datasets>=2.16.0 huggingface_hub>=0.20.0
!pip install -q pandas matplotlib seaborn tabulate tqdm python-dotenv scipy
!pip install -q requests

print('\n📦 Installing AI4Bharat NeMo fork (required for IndicConformer) — ~10 min...')
!git clone -q https://github.com/AI4Bharat/NeMo.git /content/NeMo
!cd /content/NeMo && git checkout nemo-v2 -q && bash reinstall.sh > /dev/null 2>&1 \
    && echo '✅ NeMo installed' || echo '⚠️ NeMo failed — HuggingFace fallback will be used'

print('\n✅ All dependencies installed')

In [ ]:
# Mount Google Drive (where your audio recordings are)
from google.colab import drive
drive.mount('/content/drive')

import shutil
from pathlib import Path

# Copy recordings from Drive to repo
DRIVE_RECORDINGS = '/content/drive/MyDrive/asr-shootout/recordings'
LOCAL_RECORDINGS = '/content/asr-shootout/recordings'

Path(LOCAL_RECORDINGS).mkdir(exist_ok=True)

if Path(DRIVE_RECORDINGS).exists():
    files = list(Path(DRIVE_RECORDINGS).glob('*'))
    for f in files:
        dest = Path(LOCAL_RECORDINGS) / f.name
        if not dest.exists():
            shutil.copy2(str(f), str(dest))
    print(f'✅ Copied {len(files)} files from Drive to {LOCAL_RECORDINGS}')
    print('Files found:')
    for f in sorted(Path(LOCAL_RECORDINGS).glob('*')):
        print(f'  {f.name}')
else:
    print(f'⚠️ Drive path not found: {DRIVE_RECORDINGS}')
    print('Create folder in Drive: My Drive/asr-shootout/recordings/')
    print('Then upload your audio files there and re-run this cell.')

In [ ]:
# Configure API keys — PASTE YOUR KEYS HERE
import os

# ← Paste your actual keys between the quotes
os.environ['DEEPGRAM_API_KEY'] = 'YOUR_DEEPGRAM_KEY_HERE'
os.environ['SARVAM_API_KEY']   = 'YOUR_SARVAM_KEY_HERE'

# Verify keys are set
dg_key = os.environ.get('DEEPGRAM_API_KEY', '')
sv_key = os.environ.get('SARVAM_API_KEY', '')
print(f'Deepgram key: {dg_key[:8]}...{dg_key[-4:] if len(dg_key) > 12 else "[too short]"}')
print(f'Sarvam key:   {sv_key[:8]}...{sv_key[-4:] if len(sv_key) > 12 else "[too short]"}')

In [ ]:
# Load ground truth and audio files
import sys, json
import pandas as pd
from pathlib import Path

sys.path.insert(0, '/content/asr-shootout')

RECORDINGS_DIR = Path('/content/asr-shootout/recordings')
DATA_DIR = Path('/content/asr-shootout/data')
RESULTS_DIR = Path('/content/asr-shootout/results')
RESULTS_DIR.mkdir(exist_ok=True)

# Load ground truth
with open(DATA_DIR / 'ground_truth.json') as f:
    gt_data = json.load(f)

ground_truth = {}
for rec in gt_data['recordings'] + gt_data.get('friend_recordings', []):
    ground_truth[rec['filename']] = rec

# Get all audio files
audio_files = sorted(RECORDINGS_DIR.glob('*.wav')) + \
              sorted(RECORDINGS_DIR.glob('*.mp3')) + \
              sorted(RECORDINGS_DIR.glob('*.m4a'))

print(f'✅ Loaded {len(ground_truth)} ground truth entries')
print(f'✅ Found {len(audio_files)} audio files')
print()
print('Audio files:')
for f in audio_files:
    gt = ground_truth.get(f.name, {})
    print(f'  {f.name:45s} → {gt.get("reference_transcript", "[no ground truth]")[:50]}')

## 📡 Section 1: Deepgram Nova-2 (Baseline)

In [ ]:
import src.transcribe_deepgram as dg
from src.metrics import compute_all_metrics

print('Running Deepgram Nova-2 on all recordings...')
deepgram_raw = dg.transcribe_batch([str(f) for f in audio_files], verbose=True)

deepgram_results = []
for raw, audio_path in zip(deepgram_raw, audio_files):
    gt = ground_truth.get(audio_path.name, {})
    metrics = compute_all_metrics(
        locality=gt.get('locality', ''),
        reference=gt.get('reference_transcript', ''),
        hypothesis=raw.get('transcript', ''),
        latency_ms=raw.get('latency_ms'),
        confidence=raw.get('confidence'),
    )
    metrics.update({'model': 'deepgram', 'filename': audio_path.name,
                    'condition': gt.get('condition', 'unknown'),
                    'speaker': gt.get('speaker', 'self')})
    deepgram_results.append(metrics)

df_deepgram = pd.DataFrame(deepgram_results)
print(f'\n✅ Deepgram complete')
print(f'Entity Accuracy: {df_deepgram["entity_accuracy"].mean()*100:.1f}%')
print(f'Mean WER: {df_deepgram["wer"].mean():.4f}')
print(f'Mean Latency: {df_deepgram["latency_ms"].mean():.0f}ms')

# Show failures
failures = df_deepgram[df_deepgram['entity_accuracy'] == 0][['filename', 'locality', 'hypothesis', 'recoverability']]
if len(failures):
    print(f'\n❌ Failed localities ({len(failures)}):')
    print(failures.to_string(index=False))

## 🤖 Section 2: OpenAI Whisper large-v3

In [ ]:
import src.transcribe_whisper as ws

print('Loading Whisper large-v3 (first load takes ~60s)...')
whisper_model = ws.get_model('large-v3')
print('Running inference...')

whisper_raw = ws.transcribe_batch([str(f) for f in audio_files], model_size='large-v3', verbose=True)

whisper_results = []
for raw, audio_path in zip(whisper_raw, audio_files):
    gt = ground_truth.get(audio_path.name, {})
    metrics = compute_all_metrics(
        locality=gt.get('locality', ''),
        reference=gt.get('reference_transcript', ''),
        hypothesis=raw.get('transcript', ''),
        latency_ms=raw.get('latency_ms'),
        confidence=raw.get('confidence'),
    )
    metrics.update({'model': 'whisper', 'filename': audio_path.name,
                    'condition': gt.get('condition', 'unknown'),
                    'speaker': gt.get('speaker', 'self'),
                    'is_hallucination': raw.get('is_likely_hallucination', False)})
    whisper_results.append(metrics)

df_whisper = pd.DataFrame(whisper_results)
print(f'\n✅ Whisper complete')
print(f'Entity Accuracy: {df_whisper["entity_accuracy"].mean()*100:.1f}%')
print(f'Mean WER: {df_whisper["wer"].mean():.4f}')
print(f'Mean Latency: {df_whisper["latency_ms"].mean():.0f}ms')

if 'is_hallucination' in df_whisper.columns:
    hallucinations = df_whisper[df_whisper['is_hallucination'] == True]
if len(hallucinations):
    print(f'\n⚠️ Possible hallucinations: {len(hallucinations)}')
    print(hallucinations[['filename', 'hypothesis']].to_string(index=False))

## 🇮🇳 Section 3: AI4Bharat IndicConformer

In [ ]:
import src.transcribe_indicconformer as ic

print('Loading IndicConformer (NeMo backend, may take 2-3 min)...')
indic_raw = ic.transcribe_batch([str(f) for f in audio_files], verbose=True)

indic_results = []
for raw, audio_path in zip(indic_raw, audio_files):
    gt = ground_truth.get(audio_path.name, {})
    metrics = compute_all_metrics(
        locality=gt.get('locality', ''),
        reference=gt.get('reference_transcript', ''),
        hypothesis=raw.get('transcript', ''),
        latency_ms=raw.get('latency_ms'),
        confidence=raw.get('confidence'),
    )
    metrics.update({'model': 'indicconformer', 'filename': audio_path.name,
                    'condition': gt.get('condition', 'unknown'),
                    'speaker': gt.get('speaker', 'self'),
                    'backend': raw.get('backend', 'unknown')})
    indic_results.append(metrics)

df_indic = pd.DataFrame(indic_results)
print(f'\n✅ IndicConformer complete')
backends = df_indic.get('backend', pd.Series(['unknown'] * len(df_indic))).value_counts()
print(f'Backend used: {backends.to_dict()}')
print(f'Entity Accuracy: {df_indic["entity_accuracy"].mean()*100:.1f}%')
print(f'Mean WER: {df_indic["wer"].mean():.4f}')

## 🌟 Section 4: Sarvam Saaras v2 (Indian Startup Pick)

In [ ]:
import src.transcribe_sarvam as sv

print('Running Sarvam Saaras v2...')
print('(Note: their demo uses Koramangala 5th Block as an example — this is our turf)')
sarvam_raw = sv.transcribe_batch([str(f) for f in audio_files], verbose=True)

sarvam_results = []
for raw, audio_path in zip(sarvam_raw, audio_files):
    gt = ground_truth.get(audio_path.name, {})
    metrics = compute_all_metrics(
        locality=gt.get('locality', ''),
        reference=gt.get('reference_transcript', ''),
        hypothesis=raw.get('transcript', ''),
        latency_ms=raw.get('latency_ms'),
        confidence=raw.get('confidence'),
    )
    metrics.update({'model': 'sarvam', 'filename': audio_path.name,
                    'condition': gt.get('condition', 'unknown'),
                    'speaker': gt.get('speaker', 'self')})
    sarvam_results.append(metrics)

df_sarvam = pd.DataFrame(sarvam_results)
print(f'\n✅ Sarvam complete')
print(f'Entity Accuracy: {df_sarvam["entity_accuracy"].mean()*100:.1f}%')
print(f'Mean WER: {df_sarvam["wer"].mean():.4f}')
print(f'Mean Latency: {df_sarvam["latency_ms"].mean():.0f}ms')

## 📊 Section 5: Metrics & Comparison

In [ ]:
from src.metrics import aggregate_results
from tabulate import tabulate

# Combine all results
all_dfs = [df_deepgram, df_whisper, df_indic, df_sarvam]
df_all = pd.concat(all_dfs, ignore_index=True)
df_all.to_csv(RESULTS_DIR / 'raw_transcriptions.csv', index=False)
print(f'Saved: results/raw_transcriptions.csv')

# Summary table
summaries = []
for model_name, df in [('deepgram', df_deepgram), ('whisper', df_whisper),
                        ('indicconformer', df_indic), ('sarvam', df_sarvam)]:
    agg = aggregate_results(df.to_dict('records'))
    summaries.append({
        'Model': model_name,
        'Entity Acc %': agg.get('entity_accuracy_pct'),
        'WER': agg.get('mean_wer'),
        'CER': agg.get('mean_cer'),
        'JWS': agg.get('mean_jaro_winkler'),
        'Latency ms': agg.get('mean_latency_ms'),
        'Correct': agg.get('recoverability', {}).get('correct'),
        'Recoverable': agg.get('recoverability', {}).get('recoverable'),
        'Catastrophic': agg.get('recoverability', {}).get('catastrophic'),
    })

df_summary = pd.DataFrame(summaries)
df_summary.to_csv(RESULTS_DIR / 'metrics_summary.csv', index=False)

print('\n' + '='*80)
print('SUMMARY: All Models vs Deepgram Baseline')
print('='*80)
print(tabulate(df_summary, headers='keys', tablefmt='rounded_outline', index=False))
print()
print('KEY: Entity Acc = did the locality appear correctly? (primary metric)')
print('     JWS = Jaro-Winkler Similarity on entity (recoverable vs catastrophic)')

In [ ]:
# Condition breakdown
print('\nEntity Accuracy by Condition:')
cond_breakdown = df_all.groupby(['model', 'condition'])['entity_accuracy'].mean().mul(100).round(1)
print(cond_breakdown.unstack(level=0).to_string())
cond_breakdown.reset_index().to_csv(RESULTS_DIR / 'condition_breakdown.csv', index=False)

print('\nEntity Accuracy by Speaker (accent test):')
if 'speaker' in df_all.columns:
    speaker_breakdown = df_all.groupby(['model', 'speaker'])['entity_accuracy'].mean().mul(100).round(1)
    print(speaker_breakdown.unstack(level=0).to_string())

## 📈 Section 6: Visualizations

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import numpy as np

# Style
try:
    plt.style.use('seaborn-v0_8-darkgrid')
except:
    plt.style.use('seaborn-darkgrid')
colors = {'deepgram': '#2196F3', 'whisper': '#FF9800', 'indicconformer': '#4CAF50', 'sarvam': '#9C27B0'}
models = ['deepgram', 'whisper', 'indicconformer', 'sarvam']
labels = ['Deepgram\nNova-2', 'Whisper\nlarge-v3', 'IndicConformer\n(AI4Bharat)', 'Sarvam\nSaaras v2']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('ASR Benchmark: Bangalore Locality Names in Conversational Speech', fontsize=14, fontweight='bold')

# Plot 1: Entity Accuracy (THE metric)
ax = axes[0, 0]
ea_vals = [df_all[df_all['model']==m]['entity_accuracy'].mean()*100 for m in models]
bars = ax.bar(labels, ea_vals, color=[colors[m] for m in models], alpha=0.85, edgecolor='white', linewidth=1.5)
ax.set_title('Entity Accuracy % (Primary Metric)', fontweight='bold')
ax.set_ylabel('Correct Locality %')
ax.set_ylim(0, 105)
for bar, val in zip(bars, ea_vals):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1, f'{val:.1f}%',
            ha='center', va='bottom', fontweight='bold', fontsize=11)

# Plot 2: WER comparison  
ax = axes[0, 1]
wer_vals = [df_all[df_all['model']==m]['wer'].mean()*100 for m in models]
bars = ax.bar(labels, wer_vals, color=[colors[m] for m in models], alpha=0.85, edgecolor='white', linewidth=1.5)
ax.set_title('Mean WER % (Lower = Better)', fontweight='bold')
ax.set_ylabel('Word Error Rate %')
for bar, val in zip(bars, wer_vals):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.3, f'{val:.1f}%',
            ha='center', va='bottom', fontweight='bold', fontsize=11)

# Plot 3: Latency
ax = axes[1, 0]
lat_vals = [df_all[df_all['model']==m]['latency_ms'].mean() for m in models]
bars = ax.bar(labels, lat_vals, color=[colors[m] for m in models], alpha=0.85, edgecolor='white', linewidth=1.5)
ax.set_title('Mean Latency ms (Lower = Better for Real-Time)', fontweight='bold')
ax.set_ylabel('Milliseconds')
ax.axhline(y=500, color='red', linestyle='--', alpha=0.7, label='500ms threshold')
ax.legend(fontsize=8)
for bar, val in zip(bars, lat_vals):
    if val and not np.isnan(val):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 10, f'{val:.0f}ms',
                ha='center', va='bottom', fontweight='bold', fontsize=11)

# Plot 4: Recoverability stacked bar
ax = axes[1, 1]
recover_data = {'correct': [], 'recoverable': [], 'degraded': [], 'catastrophic': []}
n = len(df_all[df_all['model']=='deepgram'])
for m in models:
    mdf = df_all[df_all['model']==m]
    for key in recover_data:
        recover_data[key].append((mdf['recoverability']==key).sum() / max(len(mdf), 1) * 100)

bottom = np.zeros(4)
rec_colors = {'correct': '#4CAF50', 'recoverable': '#8BC34A', 'degraded': '#FF9800', 'catastrophic': '#F44336'}
for key in ['correct', 'recoverable', 'degraded', 'catastrophic']:
    vals = recover_data[key]
    ax.bar(labels, vals, bottom=bottom, label=key.capitalize(), color=rec_colors[key], alpha=0.85)
    bottom += np.array(vals)

ax.set_title('Error Recoverability (Can Post-Processing Fix It?)', fontweight='bold')
ax.set_ylabel('% of samples')
ax.legend(loc='upper right', fontsize=8)
ax.set_ylim(0, 110)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'main_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/main_comparison.png')

In [ ]:
# Heatmap: Entity Accuracy by Model × Condition
fig, ax = plt.subplots(figsize=(10, 4))

conditions = df_all['condition'].unique()
heatmap_data = []
for m in models:
    row = []
    for c in conditions:
        subset = df_all[(df_all['model']==m) & (df_all['condition']==c)]
        val = subset['entity_accuracy'].mean()*100 if len(subset) > 0 else float('nan')
        row.append(val)
    heatmap_data.append(row)

hm_df = pd.DataFrame(heatmap_data, index=labels, columns=conditions)
sns.heatmap(hm_df, annot=True, fmt='.0f', cmap='RdYlGn', vmin=0, vmax=100,
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Entity Accuracy %'})
ax.set_title('Entity Accuracy (%) by Model and Noise Condition', fontweight='bold', pad=12)
ax.set_xlabel('Recording Condition')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'condition_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/condition_heatmap.png')

## 🔬 Section 7: Special Tests

In [ ]:
# HALLUCINATION TEST
# Send pure noise to all models — do they hallucinate?
print('='*60)
print('HALLUCINATION TEST: What happens with noise-only audio?')
print('='*60)
print('Production concern: a noisy/silent line should return nothing.')
print('Whisper known issue: may generate plausible-sounding fabricated text.')
print()

import numpy as np
import soundfile as sf

def generate_noise_clip(duration_s=2.0, noise_type='pink'):
    sr = 16000
    n = int(duration_s * sr)
    white = np.random.randn(n)
    noise = np.cumsum(white)
    noise = noise / np.max(np.abs(noise)) * 0.3
    path = f'/tmp/noise_{noise_type}_{duration_s}s.wav'
    sf.write(path, noise, sr)
    return path
import src.transcribe_deepgram as dg
import src.transcribe_whisper as ws
import src.transcribe_sarvam as sv

noise_path = generate_noise_clip(duration_s=2.0, noise_type='pink')
print(f'Generated pink noise clip: {noise_path}')
print()

hallucination_results = {}
# Note: IndicConformer excluded from OSS eval/special tests — NeMo inference is too slow for
# batch processing 30 samples on free Colab T4. Included in self-recordings only.
for model_name, module in [('Deepgram', dg), ('Whisper', ws), ('Sarvam', sv)]:
    result = module.transcribe_file(noise_path)
    transcript = result.get('transcript', '').strip()
    status = '🚨 HALLUCINATED' if transcript else '✅ SILENT (correct)'
    hallucination_results[model_name] = {'transcript': transcript or '[empty]', 'status': status}
    print(f'{model_name:15s}: {status}')
    if transcript:
        print(f'               → "{transcript[:80]}"')

print()
print('Note: Empty output on noise = correct behavior.')
print('      Any text output = hallucination = production risk.')

In [ ]:
# CHUNK LATENCY CURVE TEST
# How does latency scale with chunk duration?
# In telephony: VAD sends 2-10s chunks to ASR. This is the real latency.
print('='*60)
print('CHUNK LATENCY CURVE: API Latency vs Audio Duration')
print('='*60)
print('Telephony context: VAD detects utterance → sends 3-5s chunk to ASR')
print('This is the real production latency, not full-file latency.')
print()

import librosa
import soundfile as sf
import tempfile
import numpy as np

# Use the first quiet recording as our test audio
test_file = next((f for f in audio_files if 'quiet' in f.name), audio_files[0])
print(f'Test file: {test_file.name}')

audio, sr_orig = librosa.load(str(test_file), sr=16000, mono=True)
chunk_durations = [2.0, 3.0, 5.0, 10.0]
n_repeats = 3

chunk_results = {'deepgram': [], 'sarvam': []}

with tempfile.TemporaryDirectory() as tmpdir:
    for duration in chunk_durations:
        n_samples = min(int(duration * 16000), len(audio))
        chunk = audio[:n_samples]
        chunk_path = f'{tmpdir}/chunk_{duration}s.wav'
        sf.write(chunk_path, chunk, 16000)

        for model_name, module in [('deepgram', dg), ('sarvam', sv)]:
            latencies = []
            for _ in range(n_repeats):
                r = module.transcribe_file(chunk_path)
                if r.get('latency_ms'):
                    latencies.append(r['latency_ms'])

            if latencies:
                mean_lat = np.mean(latencies)
                chunk_results[model_name].append({'duration_s': duration, 'mean_latency_ms': mean_lat})
                print(f'  {model_name:10s} {duration}s chunk → {mean_lat:.0f}ms avg')

# Plot latency curve
fig, ax = plt.subplots(figsize=(8, 4))
for model_name, results in chunk_results.items():
    if results:
        x = [r['duration_s'] for r in results]
        y = [r['mean_latency_ms'] for r in results]
        ax.plot(x, y, 'o-', label=model_name.capitalize(), color=colors[model_name], linewidth=2, markersize=8)

ax.set_xlabel('Audio Chunk Duration (seconds)')
ax.set_ylabel('Mean Latency (ms)')
ax.set_title('API Latency vs Audio Chunk Duration\n(Simulates VAD-gated telephony chunks)', fontweight='bold')
ax.legend()
ax.axhline(y=500, color='red', linestyle='--', alpha=0.5, label='500ms UX threshold')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'chunk_latency_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/chunk_latency_curve.png')

## 📦 Section 8: Open-Source Datasets (MUCS + FLEURS)

In [ ]:
# MUCS 2021 — Hindi-English code-switched dataset
# This is the most relevant OSS dataset: code-switched = Hinglish = our use case
print('Loading MUCS 2021 Hindi subset (code-switched speech)...')
print('Why MUCS: spontaneous code-switched Hindi-English = closest to real Hinglish speech')
print()

from datasets import load_dataset
import tempfile, soundfile as sf

oss_results_all = []

try:
    # MUCS 2021 is not on HuggingFace — it requires manual download from openslr.org/104
    # Skipping to FLEURS which loads automatically.
    # Note in report: attempted MUCS but used FLEURS as OSS dataset.
    raise Exception("MUCS not available on HuggingFace — using FLEURS fallback")

except Exception as e:
    print(f'{e}')
    print('Trying FLEURS as fallback...')

    try:
    # MUCS 2021 is not on HuggingFace — it requires manual download from openslr.org/104
    # Skipping to FLEURS which loads automatically.
    # Note in report: attempted MUCS but used FLEURS as OSS dataset.
    raise Exception("MUCS not available on HuggingFace — using FLEURS fallback")

        fleurs = load_dataset('google/fleurs', 'hi_in', split='test', trust_remote_code=True)
        fleurs_samples = list(fleurs.select(range(20)))
        print(f'Loaded {len(fleurs_samples)} FLEURS-Hi samples (sanity check baseline)')

        with tempfile.TemporaryDirectory() as tmpdir:
            for i, sample in enumerate(fleurs_samples):
                audio_arr = sample['audio']['array']
                sr = sample['audio']['sampling_rate']
                reference = sample.get('transcription', sample.get('raw_transcription', ''))
                clip_path = f'{tmpdir}/fleurs_{i:03d}.wav'
                sf.write(clip_path, audio_arr, sr)

                # Note: IndicConformer excluded — NeMo inference too slow for batch testing here
                for model_name, module in [('deepgram', dg), ('whisper', ws), ('sarvam', sv)]:
                    r = module.transcribe_file(clip_path)
                    from src.metrics import compute_wer, compute_cer
                    oss_results_all.append({
                        'source': 'fleurs_hi', 'model': model_name,
                        'reference': reference, 'hypothesis': r.get('transcript', ''),
                        'wer': compute_wer(reference, r.get('transcript', '')),
                        'cer': compute_cer(reference, r.get('transcript', '')),
                    })

        df_oss = pd.DataFrame(oss_results_all)
        print('\nFLEURS-Hi Results (sanity check — clean read speech, expect high accuracy):')
        print(df_oss.groupby('model')[['wer', 'cer']].mean().round(4).to_string())
        df_oss.to_csv(RESULTS_DIR / 'oss_fleurs_results.csv', index=False)

    except Exception as e2:
        print(f'FLEURS also failed: {e2}')
        print('OSS dataset evaluation skipped.')

## 💾 Section 9: Export All Results

In [ ]:
# Save everything to Google Drive
import shutil

DRIVE_RESULTS = '/content/drive/MyDrive/asr-shootout/results'
Path(DRIVE_RESULTS).mkdir(parents=True, exist_ok=True)

for result_file in RESULTS_DIR.glob('*'):
    dest = Path(DRIVE_RESULTS) / result_file.name
    shutil.copy2(str(result_file), str(dest))
    print(f'Copied: {result_file.name} → Drive')

print(f'\n✅ All results saved to Google Drive: {DRIVE_RESULTS}')
print('\nFiles generated:')
for f in sorted(RESULTS_DIR.glob('*')):
    size = f.stat().st_size
    print(f'  {f.name:40s} ({size:,} bytes)')

In [ ]:
# Print final summary for copy-paste into report
print('='*70)
print('FINAL REPORT NUMBERS — COPY THESE INTO report/report_template.md')
print('='*70)

model_display = {'deepgram': 'Deepgram Nova-2', 'whisper': 'Whisper large-v3',
                 'indicconformer': 'IndicConformer', 'sarvam': 'Sarvam Saaras v2'}

models = ['deepgram', 'whisper', 'indicconformer', 'sarvam']
for model_name in models:
    mdf = df_all[df_all['model']==model_name]
    ea = mdf['entity_accuracy'].mean()*100
    wer = mdf['wer'].mean()
    cer = mdf['cer'].mean()
    jws = mdf['jaro_winkler_similarity'].mean()
    lat = mdf['latency_ms'].mean()
    rec = mdf['recoverability'].value_counts().to_dict()

    print(f'\n{model_display[model_name]}:')
    print(f'  Entity Accuracy:     {ea:.1f}%')
    print(f'  WER:                 {wer:.4f} ({wer*100:.1f}%)')
    print(f'  CER:                 {cer:.4f} ({cer*100:.1f}%)')
    print(f'  Jaro-Winkler (JWS):  {jws:.4f}')
    print(f'  Mean Latency:        {lat:.0f}ms' if lat and not pd.isna(lat) else '  Mean Latency:        N/A (local inference)')
    print(f'  Recoverability:      {rec}')